# Iowa Rest × CBraMod / S-CEReBrO + 기관 간 검증

**런타임 → 런타임 유형 변경 → T4 GPU** 후 위에서부터 실행하세요. 결정이 필요한 곳은 ✋ 표시 셀 하나뿐입니다.

| 단계 | 내용 | 통과 기준 |
|---|---|---|
| 0 | 준비 (최신 코드, 설치, S-CEReBrO) | 코드 버전 확인 통과 |
| 1 | Iowa 1명 받아 구조 확인 | 파일·채널·fs 500, 기준전극 Pz |
| ✋ 2 | 기준전극 결정 | — |
| 3 | PD 100명 다운로드 | 파일 100개 |
| 4 | CBraMod 추출·점검 | 100명 ok, 구간 24~68 (대부분 24~37), 보간 Pz |
| 5 | S-CEReBrO 추출·점검 | 동일 |
| 6 | `cv` 두 인코더 + `compare` | 같은 대상·split, EEG 미사용 모델 동일 |
| 7 | Iowa 누출 점검 | 섞으면 기준선 |
| 8 | 기관 간 검증 (C0, 4분/2분 두 조건) | — |
| 9 | 2×2 결과 표 | — |

전제: Drive `내 드라이브`에 최신 `medley_pd.zip`(compare·max_segments 포함)을 올려 두었거나, `medley_pd/medley_pd/` 코드 폴더를 최신으로 교체해 둠. PD-EEG 피처(`features/pd_eeg/cbramod`, `scerebro`)는 어제 만든 것을 그대로 사용.

## 0. 준비

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, subprocess
WORK = '/content/drive/MyDrive/medley_pd'
CODE = '/content/medley_pd'
DATA = '/content/data'
OLD  = '/content/drive/MyDrive/pd_eeg_project/ds007526'

%cd /content
!rm -rf {CODE}
if os.path.exists('/content/drive/MyDrive/medley_pd.zip'):          # 최신 zip 이 있으면 우선 사용
    !unzip -oq /content/drive/MyDrive/medley_pd.zip -d /content/
else:
    !cp -r {WORK}/medley_pd {CODE}
%cd {CODE}
ok = (os.system('grep -q "def run_compare" medley/evaluate.py') == 0 and
      os.system('grep -q "max_segments" medley/features.py') == 0)
assert ok, '코드가 최신이 아닙니다. 최신 medley_pd.zip 을 내 드라이브에 올리세요.'
print('코드 버전 OK')
!ls

os.makedirs(f'{DATA}/ds007526', exist_ok=True)
!cp {OLD}/participants.* {DATA}/ds007526/          # 기관 간 검증용 (PD-EEG 피처는 Drive 캐시 사용)

In [ ]:
!pip install -q mne openneuro-py einops huggingface_hub pyyaml timm safetensors
import torch; print('GPU:', torch.cuda.is_available())
S = (f'--set paths.data_root={DATA} --set paths.feature_root={WORK}/features '
     f'--set paths.result_root={WORK}/results --set paths.split_root={WORK}/splits')
BF = f'{WORK}/BioFoundation'
if not os.path.exists(f'{BF}/models/s_cerebro.py'):
    !git clone -q https://github.com/pulp-bio/BioFoundation "$BF"
    !cd "$BF" && git fetch -q origin 361c017ce0018141391d068e85fd6352fa32a9ef && git checkout -q FETCH_HEAD
from huggingface_hub import hf_hub_download
SCW = hf_hub_download('PulpBio/S-CEReBrO', 'S_CEReBrO.safetensors', local_dir=f'{WORK}/S-CEReBrO')
SC = f'--set encoder.repo={BF} --set encoder.weights={SCW}'
CBRAMOD_REPO, WEIGHTS = None, None
for base in [WORK, '/content/drive/MyDrive/pd_eeg_project', '/content/drive/MyDrive/mumtaz_hcmdd_project', '/content/drive/MyDrive/tdbrain_hcmdd_project']:
    if os.path.exists(f'{base}/CBraMod/models/cbramod.py') and os.path.exists(f'{base}/pretrained_weights.pth'):
        CBRAMOD_REPO, WEIGHTS = f'{base}/CBraMod', f'{base}/pretrained_weights.pth'; break
CB = f'--set encoder.repo={CBRAMOD_REPO} --set encoder.weights={WEIGHTS}'
print(CB); print(SC)

## 1. Iowa 1명 받아 구조 확인

In [ ]:
!python run.py download --dataset iowa_rest $S --set "dataset.download_include=[participants.tsv,participants.json,dataset_description.json,README,sub-001]"
!find {DATA}/ds004584/sub-001 -type f | xargs ls -la
!python run.py inspect --dataset iowa_rest $S 2>&1 | grep -v Warning | grep -v make_standard_montage | tail -8

In [ ]:
# 기록 기준전극 단서: BrainVision 헤더(.vhdr)의 Reference 관련 줄, 채널 목록에 없는 중앙선 전극
for f in glob.glob(f'{DATA}/ds004584/sub-001/**/*', recursive=True):
    if f.lower().endswith(('.vhdr', '.json')):
        print('==', f)
        for line in open(f, errors='ignore'):
            if 'ref' in line.lower() or 'ground' in line.lower():
                print('  ', line.rstrip()[:150])
import mne
from medley.eeg import FileIndex
from medley.config import load_config
cfg = load_config('iowa_rest', overrides=S.replace('--set ', '').split())
f = FileIndex(f'{DATA}/ds004584', cfg['dataset']).find('sub-001')
raw = mne.io.read_raw(f, preload=False, verbose='ERROR') if not f.endswith('.set') else mne.io.read_raw_eeglab(f, preload=False, verbose='ERROR')
names = {c.upper() for c in raw.ch_names}
print('\n채널 수', len(raw.ch_names), '| fs', raw.info['sfreq'], '| 길이', round(raw.times[-1], 1), 's')
print('채널 목록에 없는 중앙선 전극:', [c for c in ['FPZ', 'AFZ', 'FZ', 'FCZ', 'CZ', 'CPZ', 'PZ', 'POZ', 'OZ'] if c not in names])

## ✋ 2. 기준전극 결정
위 출력에서 `.vhdr`/`.json` 에 기준전극이 적혀 있으면 그 전극을, 없으면 **채널 목록에서 빠진 중앙선 전극**(BrainVision actiCAP 은 보통 FCz 기준·AFz 접지)을 씁니다. CBraMod 에는 영향이 없고, S-CEReBrO 전극 좌표에만 쓰입니다. 두 인코더 모두 같은 값으로 추출해야 하므로 여기서 한 번만 정합니다.

In [ ]:
REF = 'PZ'           # eeg.json EEGReference = Pz (sub-001 에서 확인)
R = f'--set dataset.reference={REF}'
print('Iowa 기준전극:', REF)

## 3. PD 100명 다운로드

In [ ]:
import pandas as pd
t = pd.read_csv(f'{DATA}/ds004584/participants.tsv', sep='\t')
pd_ids = t.loc[t.GROUP.eq('PD'), 'participant_id'].tolist()
inc = '[' + ','.join(['participants.tsv', 'participants.json', 'dataset_description.json', 'README'] + pd_ids) + ']'
!python run.py download --dataset iowa_rest $S --set "dataset.download_include={inc}" 2>&1 | tail -2
from medley.eeg import FileIndex
idx = FileIndex(f'{DATA}/ds004584', cfg['dataset'])
found = [s for s in pd_ids if idx.find(s)]
print(f'PD {len(pd_ids)}명 중 EEG 파일 찾음 {len(found)}명')
!du -sh {DATA}/ds004584

## 4. CBraMod 추출·점검

In [ ]:
!python run.py extract --dataset iowa_rest --encoder cbramod $S $CB $R --limit 3 2>&1 | grep -v Warning | grep -v make_standard_montage

In [ ]:
!python run.py extract --dataset iowa_rest --encoder cbramod $S $CB $R 2>&1 | grep -v Warning | grep -v make_standard_montage | tail -6
import numpy as np
def check(enc, dim):
    log = pd.read_csv(f'{WORK}/features/iowa_rest/{enc}/extract_log.csv')
    print(f'== {enc}'); print(log.status.astype(str).str[:30].value_counts().to_string())
    print('구간 수 분포:', log.n_seg.value_counts().sort_index().to_dict())
    print('보간 채널:', log.interp.fillna('').value_counts().to_dict() if 'interp' in log else '-')
    print('원 fs:', log.fs_orig.value_counts().to_dict() if 'fs_orig' in log else '-')
    bad = [s for s in log.subject if not os.path.exists(f'{WORK}/features/iowa_rest/{enc}/{s}.npz') or
           np.load(f'{WORK}/features/iowa_rest/{enc}/{s}.npz')['feat'].shape[1] != dim or
           np.isnan(np.load(f'{WORK}/features/iowa_rest/{enc}/{s}.npz')['feat']).any()]
    print('누락·모양·NaN 이상:', bad or '없음')
    print('잡음 구간 비율(중앙값):', round(float((log.n_flag / log.n_seg).median()), 3))
check('cbramod', 3800)

## 5. S-CEReBrO 추출·점검

In [ ]:
!python run.py extract --dataset iowa_rest --encoder scerebro $S $SC $R 2>&1 | grep -v Warning | grep -v make_standard_montage | tail -6
check('scerebro', 3420)

## 6. `cv` 두 인코더 + 인코더 비교

In [ ]:
!python run.py cv --dataset iowa_rest --encoder cbramod --task moca $S $R 2>&1 | grep -v Warning

In [ ]:
!python run.py cv --dataset iowa_rest --encoder scerebro --task moca $S $R 2>&1 | grep -v Warning
!python run.py compare --dataset iowa_rest --task moca --encoders cbramod scerebro $S $R

## 7. Iowa 누출 점검 (MoCA 섞기, 섞기 seed 3개)

In [ ]:
for k in [0, 1, 2]:
    t2 = pd.read_csv(f'{DATA}/ds004584/participants.tsv', sep='\t')
    mk = t2.GROUP.eq('PD') & t2.MOCA.notna()
    t2.loc[mk, 'MOCA'] = np.random.default_rng(k).permutation(t2.loc[mk, 'MOCA'].values)
    os.makedirs(f'{DATA}/ds004584_shuf', exist_ok=True)
    t2.to_csv(f'{DATA}/ds004584_shuf/participants.tsv', sep='\t', index=False)
    print(f'\n######## 섞기 seed {k}')
    !python run.py cv --dataset iowa_rest --encoder cbramod --task moca $S $R --set dataset.dir=ds004584_shuf --set paths.split_root=/content/tmp_splits_{k} --set paths.result_root=/content/tmp_results_{k} --set "evaluation.seeds=[0,1,2]" 2>&1 | sed -n '/모델별 성능/,/paired/p' | cut -c1-110

## 8. 기관 간 검증 (공통 변수 C0 = 나이·성별)
- 전체 구간: PD-EEG 약 4분(48구간) vs Iowa 대부분 2~3분(24~37구간, 최대 68)
- `_seg24`: 두 기관 모두 앞 24구간(2분)만 사용 → 기록 길이 차이 제거 (Iowa PD 전원이 24구간 이상)

In [ ]:
for enc in ['cbramod', 'scerebro']:
    for tr, te in [('pd_eeg', 'iowa_rest'), ('iowa_rest', 'pd_eeg')]:
        for seg in ['', '--set evaluation.max_segments=24']:
            print(f'\n######## {enc} | {tr} → {te} {seg}')
            !python run.py cross --train {tr} --test {te} --encoder {enc} --task moca $S $R {seg} 2>&1 | grep -v Warning | sed -n '/학습/p;/모델별 성능/,/paired 비교/p' | cut -c1-120

## 9. 2×2 결과 표 (seed 평균 예측의 MAE / Spearman)

In [ ]:
rows = []
for ds in ['pd_eeg', 'iowa_rest']:
    for enc in ['cbramod', 'scerebro']:
        p = f'{WORK}/results/{ds}/moca/{enc}/summary.csv'
        if not os.path.exists(p): continue
        s = pd.read_csv(p).set_index('model')
        for m in s.index:
            rows.append({'dataset': ds, 'encoder': enc, 'model': m, 'MAE': s.loc[m, 'avg_MAE'], 'Spearman': s.loc[m, 'avg_Spearman'], 'R2': s.loc[m, 'avg_R2']})
T = pd.DataFrame(rows)
keep = ['mean', 'eeg', 'clinical:C0', 'fusion_stack:C0', 'fusion_concat:C0', 'clinical:C1', 'fusion_stack:C1', 'clinical:C1_iowa', 'fusion_stack:C1_iowa', 'fusion_concat:C1_iowa']
W = T[T.model.isin(keep)].pivot_table(index='model', columns=['dataset', 'encoder'], values=['MAE', 'Spearman']).round(3)
print(W.to_string())
W.to_csv(f'{WORK}/results/table_2x2_moca.csv')

print('\n== 기관 간 검증')
cr = []
for p in sorted(glob.glob(f'{WORK}/results/cross/*/moca/*/summary.csv')):
    parts = p.split('/'); s = pd.read_csv(p).set_index('model')
    for m in ['mean', 'eeg', 'clinical:C0', 'fusion_stack:C0', 'fusion_concat:C0']:
        if m in s.index:
            cr.append({'direction': parts[-4], 'encoder': parts[-2], 'model': m, 'MAE': s.loc[m, 'avg_MAE'], 'Spearman': s.loc[m, 'avg_Spearman']})
C = pd.DataFrame(cr).pivot_table(index=['direction', 'model'], columns='encoder', values=['MAE', 'Spearman']).round(3)
print(C.to_string()); C.to_csv(f'{WORK}/results/table_cross_moca.csv')

## 10. 인지 저하 분류 (MoCA < 26) — 두 데이터셋 × 두 인코더
같은 피처·같은 비교 모델로 분류 과제를 돌립니다. 지표는 AUROC(주), 균형정확도, Brier. split 은 정답(0/1)으로 층화한 새 고정 split (`splits/<dataset>__moca_impaired.csv`).
로지스틱 C 선택 때문에 회귀보다 느립니다 (실행당 약 4~6분, 전체 약 20분).

In [ ]:
# 최신 코드(분류 과제 포함)로 교체 — 내 드라이브의 medley_pd.zip 을 먼저 새 것으로 바꿔 두세요
%cd /content
!rm -rf /content/medley_pd
!unzip -oq /content/drive/MyDrive/medley_pd.zip -d /content/
%cd /content/medley_pd
!test -f configs/tasks/moca_impaired.yaml && echo "코드 OK (분류 과제 포함)" || echo "⚠ 코드가 최신이 아닙니다"

In [ ]:
for ds, extra in [('pd_eeg', ''), ('iowa_rest', R)]:
    for enc in ['cbramod', 'scerebro']:
        print(f'\n######## {ds} × {enc}')
        !python run.py cv --dataset {ds} --encoder {enc} --task moca_impaired $S {extra} 2>&1 | grep -v Warning
    !python run.py compare --dataset {ds} --task moca_impaired --encoders cbramod scerebro $S {extra}